In [1]:
from pathlib import Path
import yaml

import dask.dataframe as dd
import duckdb
from wazeasy import utils, reports


def find_project_root(file="pyproject.toml"):
    current_dir = Path.cwd()
    for parent in [current_dir] + list(current_dir.parents):
        if (parent / file).is_file():
            return parent
    raise FileNotFoundError(f"{file} not found in any parent directories.")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data"

In [2]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [3]:
sample_path = DATA_PATH / "IQ_682baghdad_jams_sample.parquet"
if not sample_path.exists():
    duckdb.sql("""
    CREATE OR REPLACE SECRET secret (
        TYPE s3,
        PROVIDER credential_chain,
        CHAIN 'config;sso',
        PROFILE 'ddp_dec',
        REGION 'us-east-1'
    );
    """)

    df = duckdb.sql(
        "SELECT * FROM read_parquet('s3://wbg-waze/bq/IQ/682baghdad/jams/IQ_682baghdad_jams.00000000000[0-9].parquet')"
    ).df()

    df.to_parquet(sample_path, index=False)
else:
    # df = duckdb.sql(f"SELECT * FROM read_parquet('{sample_path}')").df()
    df = dd.read_parquet(sample_path)

In [4]:
timezone = config["Baghdad"]["timezone"]["timezone_name"]
geographies = config["Baghdad"]["geographies"]
projected_crs = config["Baghdad"]["crs"]["projected_crs"]

In [5]:
df = utils.assign_geography_to_jams(df)
utils.handle_time(df, timezone)

In [6]:
reports.run_basic_report(df, "2023-01-01", "2024-12-31")

alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

In [7]:
reports.run_geog_report(df, geographies, "2023-01-01", "2024-12-31")

Running report for Administrative Level 1


alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

Running report for Administrative Level 2


alt.VConcatChart(...)

alt.VConcatChart(...)

alt.VConcatChart(...)

Running report for Hexagons Level 7
